# Model-size LLR-prior popDMS report

This notebook is the reporting layer for the ESM-C model-size benchmark. It does
**not** run inference: `experiments/model_size_llr/submit_experiment.sh` launches
the whole pipeline on the cluster (process → GPU LLR for ESM-C 300M/600M/6B →
per-dataset `esmdms analyze` → aggregate), and this notebook loads the aggregated
`baselines.csv`, `prior_sweeps.csv`, and `summary.csv` and plots them.

Point it at a different results set with the `ESMDMS_AGGREGATE_DIR` environment
variable; it defaults to `results/model_size_llr/aggregate`.

**What is compared, per dataset:** the naive enrichment ratio, the assay's own DMS
functional score (where available), regular popDMS (no prior, gamma at the popDMS
correlation elbow), the raw ESM-C LLR of each model size used directly as a
predictor, and popDMS with a scale-matched LLR prior swept over prior strength and
regularization. Prior strength is expressed as a multiple of the **matched scale**
`s* = std(regular-popDMS coefficients) / std(LLR)`, so `1` means the prior's
coefficient spread equals popDMS's own.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

AGGREGATE_DIR = Path(os.environ.get("ESMDMS_AGGREGATE_DIR", "results/model_size_llr/aggregate"))
missing = [f for f in ("baselines.csv", "prior_sweeps.csv", "summary.csv")
           if not (AGGREGATE_DIR / f).is_file()]
if missing:
    raise RuntimeError(
        f"Missing {missing} under {AGGREGATE_DIR}. Run "
        "experiments/model_size_llr/submit_experiment.sh first, or set "
        "ESMDMS_AGGREGATE_DIR to an existing aggregate directory."
    )

baselines = pd.read_csv(AGGREGATE_DIR / "baselines.csv")
sweeps = pd.read_csv(AGGREGATE_DIR / "prior_sweeps.csv")
summary = pd.read_csv(AGGREGATE_DIR / "summary.csv")

PRIMARY_CUTOFF = 0
AUC = f"auc_stars_{PRIMARY_CUTOFF}"
DATASETS = sorted(baselines["dataset"].unique())
MODEL_SIZES = ["300M", "600M", "6B"]

# Ordinal model size -> single-hue blue ramp (light = small, dark = large).
SIZE_COLOR = {"300M": "#9ecae1", "600M": "#4292c6", "6B": "#08519c"}
# Colorblind-safe (Okabe-Ito) accents for the non-ESM baselines.
BASELINE_COLOR = {
    "Enrichment ratio": "#999999",
    "DMS functional score": "#000000",
    "Regular popDMS": "#E69F00",
    # popDMS shrunk toward the per-site mean coefficient (computed below).
    "Site-mean prior popDMS": "#009E73",
}

def size_of(label):
    # "ESM-C 300M LLR" / "Raw ESM-C 300M LLR" -> "300M"
    for token in str(label).split():
        if token in SIZE_COLOR:
            return token
    return None

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
                     "figure.dpi": 110, "font.size": 10})
print(f"Loaded {len(DATASETS)} datasets from {AGGREGATE_DIR}")
summary

## Site-mean-prior popDMS

An extra popDMS variant: solve regular popDMS at its elbow gamma, take the **mean
selection coefficient over all mutations at each site** (the per-column mean of the
mutation-by-site coefficient matrix), and re-solve using that per-site mean as the
popDMS prior — shrinking every substitution toward its site's average effect. The
prior enters with weight `precision = gamma x n_replicates`, so gamma controls the
shrinkage strength; it is swept over the popDMS grid and reported at the **best
primary-cutoff AUC**, the same operating-point rule used for the LLR priors (an
elbow/consistency rule is unusable here because consistency degenerates to 1.0 as
gamma grows and all replicates collapse onto the fixed prior).

This is the one place the report leaves the aggregated CSVs and re-solves from the
raw datasets, so the result is **cached** to `site_mean_prior.csv`; delete that file
to force a recompute.

In [ ]:
# Compute (or load from cache) the site-mean-prior popDMS baseline, then merge it
# into `baselines` so every downstream baseline-driven plot picks it up.
SITE_MEAN_METHOD = "Site-mean prior popDMS"
CUTOFFS = [0, 1, 2, 3]
_site_mean_cache = AGGREGATE_DIR / "site_mean_prior.csv"

def _compute_site_mean_prior():
    from esmdms.schema import Dataset
    from esmdms.inference import substitution_basis, build_problem
    from esmdms.regularization import popdms_gamma_grid
    from esmdms.workflow import _metric_columns

    config_dir = AGGREGATE_DIR.parent / "configs"
    rows = []
    for dataset in DATASETS:
        reg = baselines[(baselines["dataset"] == dataset) & (baselines["method"] == "Regular popDMS")]
        config_path = config_dir / f"{dataset}.json"
        if reg.empty or not config_path.is_file():
            print(f"[{dataset}] no Regular popDMS row or config; site-mean prior skipped.")
            continue
        reg_gamma = float(reg["gamma"].iloc[0])
        config = json.loads(config_path.read_text())
        ds = Dataset.load(config["datasets"][0]["path"])
        basis = substitution_basis(ds)
        problem = build_problem(ds, basis)

        # Per-site mean of the regular-popDMS coefficients -> prior mean vector.
        coeffs = problem.solve(gamma=reg_gamma).joint_coefficients
        positions = np.array([name.split(":")[0] for name in basis.feature_names])
        site_mean = np.zeros_like(coeffs)
        for position in np.unique(positions):
            mask = positions == position
            site_mean[mask] = coeffs[mask].mean()

        # Sweep the shrinkage strength (gamma); keep the best primary-cutoff AUC,
        # tie-broken toward weaker shrinkage (smaller gamma).
        best = None
        for gamma in popdms_gamma_grid(ds):
            result = problem.solve(gamma=float(gamma), prior_values=site_mean)
            metrics = _metric_columns(ds, result.fitness(), CUTOFFS)
            key = (metrics[f"auc_stars_{PRIMARY_CUTOFF}"], -float(gamma))
            if best is None or key > best[0]:
                best = (key, float(gamma), result.cross_replicate_consistency, metrics)
        _, gamma, consistency, metrics = best
        rows.append({"dataset": dataset, "method": SITE_MEAN_METHOD, "gamma": gamma,
                     "cross_replicate_consistency": consistency, **metrics})
    return pd.DataFrame(rows)

if _site_mean_cache.is_file():
    site_mean_prior = pd.read_csv(_site_mean_cache)
    print(f"Loaded cached site-mean prior from {_site_mean_cache}")
else:
    site_mean_prior = _compute_site_mean_prior()
    site_mean_prior.to_csv(_site_mean_cache, index=False)
    print(f"Computed and cached site-mean prior to {_site_mean_cache}")

baselines = pd.concat([baselines, site_mean_prior], ignore_index=True)
site_mean_prior[["dataset", "gamma", "cross_replicate_consistency", AUC]]


## Predictor comparison by dataset (ClinVar AUC)

One panel per dataset. Bars are the assay-oriented ClinVar AUC at review-star
cutoff 0. Color encodes ESM-C model size (blue ramp); **hatched** bars are the raw
LLR used directly, **solid** ESM-size bars are popDMS with that model's
scale-matched prior (best over the sweep). Grey/orange/black are the non-ESM
baselines. The dashed line is chance (0.5).

In [ ]:
def dataset_bars(dataset, auc_col):
    rows, colors, hatches = [], [], []
    bl = baselines[baselines["dataset"] == dataset].set_index("method")[auc_col]
    for method in ("Enrichment ratio", "DMS functional score", "Regular popDMS", "Site-mean prior popDMS"):
        if method in bl.index and np.isfinite(bl[method]):
            rows.append((method, bl[method])); colors.append(BASELINE_COLOR[method]); hatches.append("")
    for size in MODEL_SIZES:
        raw = f"Raw ESM-C {size} LLR"
        if raw in bl.index and np.isfinite(bl[raw]):
            rows.append((f"Raw LLR {size}", bl[raw])); colors.append(SIZE_COLOR[size]); hatches.append("////")
    for size in MODEL_SIZES:
        prior = f"ESM-C {size} LLR"
        bp = sweeps[(sweeps["dataset"] == dataset) & (sweeps["prior"] == prior)][auc_col].max()
        if np.isfinite(bp):
            rows.append((f"Prior popDMS {size}", bp)); colors.append(SIZE_COLOR[size]); hatches.append("")
    return rows, colors, hatches

def plot_bars_by_dataset(cutoff):
    """One panel per dataset of ClinVar AUC at the given review-star cutoff.

    The y-axis is zoomed to each panel's own data so the near-ceiling
    differences between the top predictors are legible, and every bar is
    annotated with its value."""
    auc_col = f"auc_stars_{cutoff}"
    ncol = 2
    nrow = int(np.ceil(len(DATASETS) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.7 * nrow), squeeze=False)
    for ax, dataset in zip(axes.flat, DATASETS):
        rows, colors, hatches = dataset_bars(dataset, auc_col)
        if not rows:
            ax.set_visible(False); continue
        # Sort predictors by AUC (best on the left) so the ranking reads left-to-right.
        order = sorted(range(len(rows)), key=lambda k: rows[k][1], reverse=True)
        rows = [rows[k] for k in order]
        colors = [colors[k] for k in order]
        hatches = [hatches[k] for k in order]
        labels = [r[0] for r in rows]; values = [r[1] for r in rows]
        x = np.arange(len(rows))
        bars = ax.bar(x, values, color=colors, width=0.72, edgecolor="white")
        for bar, h in zip(bars, hatches):
            if h: bar.set_hatch(h)
        # Zoom to the data so top-of-range differences are distinguishable.
        lo = max(0.0, min(values) - 0.05)
        hi = max(values) + 0.06
        ax.set_ylim(lo, hi)
        if lo < 0.5 < hi:
            ax.axhline(0.5, color="#C0392B", ls="--", lw=1)
        ax.bar_label(bars, labels=[f"{v:.3f}" for v in values],
                     rotation=90, padding=2, fontsize=6.5)
        ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("ClinVar AUC")
        ax.set_title(dataset, fontsize=10)
        # Label counts of ClinVar variants available at this cutoff (same for all methods).
        brow = baselines[baselines["dataset"] == dataset].iloc[0]
        n_ben = int(brow[f"n_benign_stars_{cutoff}"]); n_path = int(brow[f"n_pathogenic_stars_{cutoff}"])
        ax.text(0.98, 0.96, f"benign n={n_ben}\npathogenic n={n_path}",
                transform=ax.transAxes, ha="right", va="top", fontsize=7.5,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.7", alpha=0.85))
    for ax in axes.flat[len(DATASETS):]:
        ax.set_visible(False)
    size_handles = [Patch(facecolor=SIZE_COLOR[s], label=f"ESM-C {s}") for s in MODEL_SIZES]
    type_handles = [Patch(facecolor="#cccccc", hatch="////", label="raw LLR"),
                    Patch(facecolor="#cccccc", label="prior popDMS (best)")]
    fig.legend(handles=size_handles + type_handles, loc="upper center",
               ncol=5, bbox_to_anchor=(0.5, 1.03), fontsize=9, frameon=False)
    fig.suptitle(f"ClinVar AUC — review-star cutoff ≥ {cutoff}", y=1.06, fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    plt.show()

plot_bars_by_dataset(PRIMARY_CUTOFF)


## Predictor comparison at every ClinVar review-star cutoff

The same zoomed, value-labeled bar comparison at each review-star confidence
cutoff (≥ 0, 1, 2, 3 stars). Higher cutoffs keep only higher-confidence
ClinVar labels, so the benign/pathogenic counts shrink as the cutoff rises.

In [ ]:
for cutoff in (0, 1, 2, 3):
    plot_bars_by_dataset(cutoff)


## Cross-replicate consistency by method

Ranking of the mean pairwise cross-replicate consistency of each popDMS-based
method, per dataset (best on the left). This is only defined for methods that are
solved per replicate: **regular popDMS** and **prior popDMS** — the enrichment
ratio, the assay DMS score, and the raw ESM-C LLR are not replicate-solved and so
have no consistency. Each prior popDMS is reported at the same best-AUC operating
point shown in the AUC bars above; the prior generally raises consistency over
no-prior popDMS.

In [ ]:
def consistency_rows(dataset):
    rows, colors = [], []
    bl = baselines[baselines["dataset"] == dataset].set_index("method")["cross_replicate_consistency"]
    for baseline in ("Regular popDMS", "Site-mean prior popDMS"):
        if baseline in bl.index and np.isfinite(bl[baseline]):
            rows.append((baseline, bl[baseline])); colors.append(BASELINE_COLOR[baseline])
    for size in MODEL_SIZES:
        prior = f"ESM-C {size} LLR"
        frame = sweeps[(sweeps["dataset"] == dataset) & (sweeps["prior"] == prior)]
        if frame.empty:
            continue
        best = frame.loc[frame[AUC].idxmax()]  # operating point reported in the AUC bars
        if np.isfinite(best["cross_replicate_consistency"]):
            rows.append((f"Prior popDMS {size}", float(best["cross_replicate_consistency"])))
            colors.append(SIZE_COLOR[size])
    order = sorted(range(len(rows)), key=lambda k: rows[k][1], reverse=True)
    return [rows[k] for k in order], [colors[k] for k in order]

ncol = 2
nrow = int(np.ceil(len(DATASETS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.4 * nrow), squeeze=False)
for ax, dataset in zip(axes.flat, DATASETS):
    rows, colors = consistency_rows(dataset)
    if not rows:
        ax.set_visible(False); continue
    labels = [r[0] for r in rows]; values = [r[1] for r in rows]
    x = np.arange(len(rows))
    bars = ax.bar(x, values, color=colors, width=0.6, edgecolor="white")
    lo = max(0.0, min(values) - 0.05); hi = min(1.0, max(values) + 0.05)
    ax.set_ylim(lo, hi)
    ax.bar_label(bars, labels=[f"{v:.3f}" for v in values], padding=2, fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel("cross-replicate consistency")
    ax.set_title(dataset, fontsize=10)
for ax in axes.flat[len(DATASETS):]:
    ax.set_visible(False)
size_handles = [Patch(facecolor=SIZE_COLOR[s], label=f"prior popDMS {s}") for s in MODEL_SIZES]
size_handles = [Patch(facecolor=BASELINE_COLOR["Regular popDMS"], label="regular popDMS"),
                Patch(facecolor=BASELINE_COLOR["Site-mean prior popDMS"], label="site-mean prior popDMS")] + size_handles
fig.legend(handles=size_handles, loc="upper center", ncol=5,
           bbox_to_anchor=(0.5, 1.02), fontsize=9, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.98))
plt.show()


### Cross-replicate consistency dot plot

The same consistency ranking as a dot plot: one column per dataset (ordered
left-to-right by the best consistency any method reaches), a marker per method.
Two notes on how this differs from a from-scratch replicate recomputation, because
this notebook reads only the aggregated results and never re-solves:

- The metric is the pipeline's stored **mean pairwise Pearson r** between replicate
  selection-coefficient vectors (`cross_replicate_consistency`), not a re-derived
  Spearman rho, and replicate counts are not carried in the aggregate.
- Consistency exists only for the **replicate-solved** methods — regular popDMS and
  prior popDMS (each prior taken at its best-AUC `alpha`/`gamma`, matching the AUC
  bars). The enrichment ratio, DMS score, and raw LLR are not replicate-solved and
  so have none.

In [ ]:
def short_name(dataset):
    parts = dataset.split("_")
    return parts[1] if len(parts) > 1 else dataset

# Assemble one consistency value per (dataset, method) from the aggregate.
dot_rows = []
for dataset in DATASETS:
    for baseline in ("Regular popDMS", "Site-mean prior popDMS"):
        r = baselines[(baselines["dataset"] == dataset) & (baselines["method"] == baseline)]
        if not r.empty and np.isfinite(r["cross_replicate_consistency"].iloc[0]):
            dot_rows.append({"dataset": dataset, "method": baseline,
                             "consistency": float(r["cross_replicate_consistency"].iloc[0]),
                             "gamma": float(r["gamma"].iloc[0])})
    for size in MODEL_SIZES:
        prior = f"ESM-C {size} LLR"
        frame = sweeps[(sweeps["dataset"] == dataset) & (sweeps["prior"] == prior)]
        if frame.empty:
            continue
        best = frame.sort_values([AUC, "alpha", "gamma"], ascending=[False, True, True]).iloc[0]
        if np.isfinite(best["cross_replicate_consistency"]):
            dot_rows.append({"dataset": dataset, "method": f"Prior popDMS {size}",
                             "consistency": float(best["cross_replicate_consistency"]),
                             "alpha": float(best["alpha"]), "gamma": float(best["gamma"])})
cross_replicate = pd.DataFrame(dot_rows)

# Order datasets by the best consistency any method reaches (descending).
best_per_dataset = cross_replicate.groupby("dataset")["consistency"].max().sort_values(ascending=False)
dataset_order = list(best_per_dataset.index)
xpos = {d: i for i, d in enumerate(dataset_order)}

method_order = [f"Prior popDMS {s}" for s in MODEL_SIZES] + ["Regular popDMS", "Site-mean prior popDMS"]
MARKERS = {"Prior popDMS 300M": "o", "Prior popDMS 600M": "^",
           "Prior popDMS 6B": "s", "Regular popDMS": "X", "Site-mean prior popDMS": "D"}
COLORS = {**{f"Prior popDMS {s}": SIZE_COLOR[s] for s in MODEL_SIZES},
          "Regular popDMS": BASELINE_COLOR["Regular popDMS"],
          "Site-mean prior popDMS": BASELINE_COLOR["Site-mean prior popDMS"]}

# Dodge each method horizontally within its dataset column so overlapping
# points stay separable.
dodge = np.linspace(-0.22, 0.22, len(method_order))
method_dodge = dict(zip(method_order, dodge))

fig, ax = plt.subplots(figsize=(10, 5.5))
for method in method_order:
    sub = cross_replicate[cross_replicate["method"] == method]
    if sub.empty:
        continue
    ax.scatter([xpos[d] + method_dodge[method] for d in sub["dataset"]], sub["consistency"],
               marker=MARKERS[method], color=COLORS[method], s=130,
               edgecolor="black", linewidth=0.6, zorder=3, label=method)
ax.set_xticks(range(len(dataset_order)))
ax.set_xticklabels([short_name(d) for d in dataset_order], rotation=30, ha="right", rotation_mode="anchor")
ax.set_xlabel("Dataset (ordered by best cross-replicate consistency)")
ax.set_ylabel("Mean pairwise cross-replicate consistency (Pearson r)")
ax.set_title("Cross-replicate consistency by method, ordered by best consistency", fontsize=11)
ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
ax.margins(x=0.08)
ax.legend(title="Method", loc="center left", bbox_to_anchor=(1.02, 0.5),
          facecolor="white", framealpha=0.9)
fig.tight_layout(rect=(0, 0, 0.82, 1))
plt.show()

cross_replicate.to_csv(AGGREGATE_DIR / "cross_replicate_consistency_dotplot.csv", index=False)
cross_replicate


## Correlation with the assay functional score (Spearman ρ)

Only datasets and methods with a finite functional score appear (BRCA2 has none).
Same color/hatch scheme as above.

In [ ]:
def spearman_rows(dataset):
    bl = baselines[baselines["dataset"] == dataset].set_index("method")["spearman_rho"]
    order, colors, hatches = [], [], []
    for method in ("Enrichment ratio", "Regular popDMS"):
        if method in bl.index and np.isfinite(bl[method]):
            order.append((method, bl[method])); colors.append(BASELINE_COLOR[method]); hatches.append("")
    for size in MODEL_SIZES:
        raw = f"Raw ESM-C {size} LLR"
        if raw in bl.index and np.isfinite(bl[raw]):
            order.append((f"Raw LLR {size}", bl[raw])); colors.append(SIZE_COLOR[size]); hatches.append("////")
    return order, colors, hatches

scored = [d for d in DATASETS
          if baselines[(baselines["dataset"] == d)]["spearman_rho"].notna().any()]
if not scored:
    print("No functional scores available in this result set.")
else:
    fig, axes = plt.subplots(1, len(scored), figsize=(3.6 * len(scored), 3.6), squeeze=False)
    for ax, dataset in zip(axes.flat, scored):
        rows, colors, hatches = spearman_rows(dataset)
        x = np.arange(len(rows))
        bars = ax.bar(x, [r[1] for r in rows], color=colors, edgecolor="white", width=0.7)
        for bar, h in zip(bars, hatches):
            if h: bar.set_hatch(h)
        ax.axhline(0, color="#666", lw=0.8)
        ax.set_xticks(x); ax.set_xticklabels([r[0] for r in rows], rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Spearman ρ vs functional score"); ax.set_title(dataset, fontsize=10)
    fig.tight_layout(); plt.show()

## Does the prior help, and at what strength?

For each dataset and model size, the best achievable AUC (over gamma) at each prior
strength, on a log2 axis of the **matched-scale multiple** (`1` = matched to the
popDMS coefficient spread). The grey dashed line is the `alpha = 0` no-prior
control; the square marker is the **unscaled raw-LLR** magnitude. A prior that helps
rises above its control line somewhere near `1`.

In [ ]:
scaled = sweeps[(sweeps["scale_multiple"] > 0) & (~sweeps["unscaled_raw_llr"])]
best_scaled = scaled.groupby(["dataset", "prior", "scale_multiple"])[AUC].max().reset_index()
controls = sweeps[sweeps["alpha"] == 0.0].groupby(["dataset", "prior"])[AUC].max()
unscaled = sweeps[sweeps["unscaled_raw_llr"]].groupby(["dataset", "prior"]).agg(
    scale_multiple=("scale_multiple", "first"), auc=(AUC, "max"))

ncol = 2
nrow = int(np.ceil(len(DATASETS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.4 * nrow), squeeze=False)
for ax, dataset in zip(axes.flat, DATASETS):
    for size in MODEL_SIZES:
        prior = f"ESM-C {size} LLR"
        line = best_scaled[(best_scaled["dataset"] == dataset) & (best_scaled["prior"] == prior)]
        if line.empty:
            continue
        line = line.sort_values("scale_multiple")
        ax.plot(line["scale_multiple"], line[AUC], marker="o", color=SIZE_COLOR[size],
                lw=2, label=f"ESM-C {size}")
        if (dataset, prior) in controls.index:
            ax.axhline(controls[(dataset, prior)], color=SIZE_COLOR[size], ls="--", lw=1, alpha=0.7)
        if (dataset, prior) in unscaled.index:
            u = unscaled.loc[(dataset, prior)]
            ax.scatter(u["scale_multiple"], u["auc"], color=SIZE_COLOR[size],
                       marker="s", s=55, zorder=5, edgecolor="white")
    ax.set_xscale("log", base=2)
    ax.axhline(0.5, color="#C0392B", ls=":", lw=1)
    ax.set_xlabel("prior strength (× matched scale s*)"); ax.set_ylabel("best ClinVar AUC")
    ax.set_title(dataset, fontsize=10); ax.legend(fontsize=8, title="dashed = no-prior control")
for ax in axes.flat[len(DATASETS):]:
    ax.set_visible(False)
fig.tight_layout(); plt.show()

## Regularization sensitivity (alpha × gamma surface)

The full surface behind the summary: best AUC as a function of gamma
(regularization / prior precision), one line per matched-scale multiple, per dataset
and model size.

In [ ]:
for dataset in DATASETS:
    priors = sorted(sweeps[sweeps["dataset"] == dataset]["prior"].unique())
    if not priors:
        continue
    fig, axes = plt.subplots(1, len(priors), figsize=(4.6 * len(priors), 3.6), squeeze=False)
    for ax, prior in zip(axes.flat, priors):
        frame = sweeps[(sweeps["dataset"] == dataset) & (sweeps["prior"] == prior)]
        surface = frame.pivot_table(index="gamma", columns="scale_multiple", values=AUC)
        cmap = plt.cm.viridis(np.linspace(0, 1, surface.shape[1]))
        for color, col in zip(cmap, surface.columns):
            ax.plot(surface.index, surface[col], color=color, lw=1.4,
                    label=f"{col:g}×" if col > 0 else "0")
        ax.set_xscale("log"); ax.axhline(0.5, color="#C0392B", ls=":", lw=1)
        ax.set_xlabel("gamma"); ax.set_ylabel("ClinVar AUC")
        ax.set_title(f"{dataset}\n{prior}", fontsize=9)
        ax.legend(fontsize=7, title="× s*", ncol=2)
    fig.tight_layout(); plt.show()

## Alpha × gamma AUC heatmaps

For every dataset (rows) and ESM-C prior (columns), the ClinVar AUC (cutoff 0) as
a joint function of prior strength `alpha` (y-axis) and regularization `gamma`
(x-axis). Colour is AUC on a shared scale across all panels — brighter is
better. This is the full surface the line plots above summarize.

In [ ]:
prior_list = sorted(sweeps["prior"].unique())
vmin = float(np.nanmin(sweeps[AUC])); vmax = float(np.nanmax(sweeps[AUC]))
fig, axes = plt.subplots(len(DATASETS), len(prior_list),
                         figsize=(3.9 * len(prior_list), 3.1 * len(DATASETS)),
                         squeeze=False)
im = None
for i, dataset in enumerate(DATASETS):
    for j, prior in enumerate(prior_list):
        ax = axes[i][j]
        frame = sweeps[(sweeps["dataset"] == dataset) & (sweeps["prior"] == prior)]
        if frame.empty:
            ax.set_visible(False); continue
        piv = frame.pivot_table(index="alpha", columns="gamma", values=AUC, aggfunc="max")
        piv = piv.sort_index().reindex(sorted(piv.columns), axis=1)
        im = ax.imshow(piv.values, aspect="auto", origin="lower",
                       cmap="viridis", vmin=vmin, vmax=vmax)
        # Thin the gamma ticks (~8) so the labels stay legible.
        gtick = max(1, len(piv.columns) // 8)
        gpos = range(0, len(piv.columns), gtick)
        ax.set_xticks(list(gpos))
        ax.set_xticklabels([f"{piv.columns[k]:.0e}" for k in gpos], rotation=90, fontsize=6)
        ax.set_yticks(range(len(piv.index)))
        ax.set_yticklabels([f"{a:.2g}" for a in piv.index], fontsize=6)
        if i == len(DATASETS) - 1:
            ax.set_xlabel("gamma", fontsize=8)
        ax.set_ylabel((f"{dataset}\n" if j == 0 else "") + "alpha", fontsize=8)
        if i == 0:
            ax.set_title(prior, fontsize=9)
if im is not None:
    fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02,
                 label=f"ClinVar AUC (cutoff {PRIMARY_CUTOFF})")
plt.show()


## Selected regularization and matched scale

For reproducibility: the popDMS elbow gamma chosen for regular popDMS, and the
matched scale `s*` with the coefficient/prior standard deviations that define it,
per dataset and model size.

In [ ]:
elbow = (baselines[baselines["method"] == "Regular popDMS"]
         .set_index("dataset")["gamma"].rename("elbow_gamma"))
scale_tbl = (sweeps.groupby(["dataset", "prior"])
             .agg(matched_scale=("matched_scale", "first"),
                  sigma_coeff=("sigma_coeff", "first"),
                  sigma_prior=("sigma_prior", "first"))
             .reset_index())
scale_tbl["model_size"] = scale_tbl["prior"].map(size_of)
scale_tbl = scale_tbl.merge(elbow, left_on="dataset", right_index=True, how="left")
scale_tbl[["dataset", "model_size", "elbow_gamma", "matched_scale",
           "sigma_coeff", "sigma_prior"]].sort_values(["dataset", "model_size"])